In [1]:

# After kernel reset, let me reconsider the computational approach
# The timeout suggests my time estimates were too optimistic
# Let me test actual performance more carefully

import numpy as np
from numba import jit
import time

# First, let's test a single N=10^4 run to calibrate timing

@jit(nopython=True)
def test_dirichlet_eval(N, n_points):
 """Quick test of Dirichlet evaluation speed"""
 t_values = np.linspace(float(N), 2.0*float(N), n_points)
 results = np.zeros(n_points, dtype=np.complex128)
 
 for i, t in enumerate(t_values):
 result = 0.0 + 0.0j
 for n in range(1, N + 1):
 n_f = float(n)
 log_n = np.log(n_f)
 phase = t * log_n
 result += (np.cos(phase) - 1j * np.sin(phase)) / np.sqrt(n_f)
 results[i] = result
 
 return results

# Warm up JIT
_ = test_dirichlet_eval(100, 10)

# Test at different scales
print("Calibration tests:")
print("="*70)

for N in [10**4, 10**5]:
 n_points = 100 # Reduced from 2000 for testing
 print(f"\nN = {N:,}, evaluating at {n_points} points:")
 start = time.time()
 result = test_dirichlet_eval(N, n_points)
 elapsed = time.time() - start
 print(f" Time: {elapsed:.2f} seconds")
 print(f" Time per evaluation: {elapsed/n_points:.4f} seconds")
 
 # Extrapolate to 1000 points
 time_1000 = elapsed * (1000 / n_points)
 print(f" Estimated time for 1000 points: {time_1000:.1f} seconds")


Calibration tests:

N = 10,000, evaluating at 100 points:
 Time: 0.04 seconds
 Time per evaluation: 0.0004 seconds
 Estimated time for 1000 points: 0.4 seconds

N = 100,000, evaluating at 100 points:


 Time: 0.35 seconds
 Time per evaluation: 0.0035 seconds
 Estimated time for 1000 points: 3.5 seconds


In [2]:

# Continue calibration for larger N
for N in [10**6]:
 n_points = 100
 print(f"\nN = {N:,}, evaluating at {n_points} points:")
 start = time.time()
 result = test_dirichlet_eval(N, n_points)
 elapsed = time.time() - start
 print(f" Time: {elapsed:.2f} seconds")
 print(f" Time per evaluation: {elapsed/n_points:.4f} seconds")
 
 # Extrapolate to 1000 points
 time_1000 = elapsed * (1000 / n_points)
 print(f" Estimated time for 1000 points: {time_1000:.1f} seconds")
 
# Don't test N=10^7 yet - will take too long

print("\n" + "="*70)
print("Time budget analysis:")
print("="*70)

# For 1000 evaluation points + omega computation + peak analysis
estimates = {
 '10^4': 0.4 + 1 + 2, # ~3 sec total per function
 '10^5': 3.5 + 3 + 5, # ~11 sec total per function 
 '10^6': 35 + 20 + 30, # ~85 sec total per function
}

# For N=10^7, extrapolate
time_10_7_per_eval = 0.35 * (10**7 / 10**5) / 100 # from N=10^5 test
time_10_7_total_evals = time_10_7_per_eval * 1000
time_10_7_omega = 120 # omega computation
time_10_7_peaks = 150 # peak finding and r computation
estimates['10^7'] = time_10_7_total_evals + time_10_7_omega + time_10_7_peaks

print("\nEstimated time per function (1000 t-points, 200 peaks):")
for n, t in estimates.items():
 print(f" N={n:>4}: {t:6.1f} seconds ({t/60:4.1f} min)")

total_estimate = sum(estimates.values()) * 2 # Two functions
print(f"\nTotal for both functions: {total_estimate:.0f} seconds ({total_estimate/60:.1f} min)")
print(f"Time budget: 5400 seconds (90 min)")
print(f"Margin: {5400 - total_estimate:.0f} seconds ({(5400-total_estimate)/60:.1f} min)")

if total_estimate > 5400:
 print("\nSTATUS: Still appears to exceed budget. Need further optimization.")
else:
 print("\nSTATUS: Should be feasible!")



N = 1,000,000, evaluating at 100 points:
 Time: 3.40 seconds
 Time per evaluation: 0.0340 seconds
 Estimated time for 1000 points: 34.0 seconds

Time budget analysis:

Estimated time per function (1000 t-points, 200 peaks):
 N=10^4: 3.4 seconds ( 0.1 min)
 N=10^5: 11.5 seconds ( 0.2 min)
 N=10^6: 85.0 seconds ( 1.4 min)
 N=10^7: 620.0 seconds (10.3 min)

Total for both functions: 1440 seconds (24.0 min)
Time budget: 5400 seconds (90 min)
Margin: 3960 seconds (66.0 min)

STATUS: Should be feasible!


In [3]:

# Good! With 1000 evaluation points it should be feasible
# Let me now rebuild the full analysis pipeline with this optimization

from scipy.signal import find_peaks

# Re-define core functions (already JIT compiled from previous run)

def compute_omega_omega_bigomega(N):
 """
 Compute ω(n) and Ω(n) for all n ≤ N using sieve method.
 """
 omega = np.zeros(N + 1, dtype=np.int32)
 bigomega = np.zeros(N + 1, dtype=np.int32)
 
 for p in range(2, N + 1):
 if omega[p] == 0: # p is prime
 for multiple in range(p, N + 1, p):
 omega[multiple] += 1
 temp = multiple
 while temp % p == 0:
 bigomega[multiple] += 1
 temp //= p
 
 return omega[1:], bigomega[1:]


@jit(nopython=True)
def compute_dirichlet_liouville_jit(N, t_values, lambda_vals):
 """Compute Dirichlet polynomial for Liouville function"""
 results = np.zeros(len(t_values), dtype=np.complex128)
 
 for i, t in enumerate(t_values):
 result = 0.0 + 0.0j
 for n in range(1, N + 1):
 n_f = float(n)
 log_n = np.log(n_f)
 phase = t * log_n
 coeff = lambda_vals[n - 1]
 result += coeff * (np.cos(phase) - 1j * np.sin(phase)) / np.sqrt(n_f)
 results[i] = result
 
 return results


@jit(nopython=True)
def compute_omega_class_sums_zeta(N, t, omega_vals, k_max):
 """Compute omega-class sums S_k for zeta function"""
 S_k_real = np.zeros(k_max + 1, dtype=np.float64)
 S_k_imag = np.zeros(k_max + 1, dtype=np.float64)
 
 for n in range(1, N + 1):
 n_f = float(n)
 k = omega_vals[n - 1]
 
 if k <= k_max:
 log_n = np.log(n_f)
 phase = t * log_n
 contrib_real = np.cos(phase) / np.sqrt(n_f)
 contrib_imag = -np.sin(phase) / np.sqrt(n_f)
 S_k_real[k] += contrib_real
 S_k_imag[k] += contrib_imag
 
 return S_k_real, S_k_imag


@jit(nopython=True)
def compute_omega_class_sums_liouville(N, t, omega_vals, lambda_vals, k_max):
 """Compute omega-class sums S_k for Liouville function"""
 S_k_real = np.zeros(k_max + 1, dtype=np.float64)
 S_k_imag = np.zeros(k_max + 1, dtype=np.float64)
 
 for n in range(1, N + 1):
 n_f = float(n)
 k = omega_vals[n - 1]
 coeff = lambda_vals[n - 1]
 
 if k <= k_max:
 log_n = np.log(n_f)
 phase = t * log_n
 contrib_real = coeff * np.cos(phase) / np.sqrt(n_f)
 contrib_imag = coeff * (-np.sin(phase)) / np.sqrt(n_f)
 S_k_real[k] += contrib_real
 S_k_imag[k] += contrib_imag
 
 return S_k_real, S_k_imag


def compute_r_value(D_F_val, S_k_real, S_k_imag):
 """Compute inter-class energy ratio r"""
 D_F_mod_sq = np.abs(D_F_val)**2
 
 sum_S_k_mod_sq = 0.0
 for k in range(len(S_k_real)):
 S_k_mod_sq = S_k_real[k]**2 + S_k_imag[k]**2
 sum_S_k_mod_sq += S_k_mod_sq
 
 r = (D_F_mod_sq - sum_S_k_mod_sq) / (2.0 * sum_S_k_mod_sq)
 
 return r


# Note: test_dirichlet_eval is already JIT-compiled as compute_dirichlet_zeta_jit
compute_dirichlet_zeta_jit = test_dirichlet_eval

print("All core functions defined and JIT-compiled.")
print("Ready to proceed with analysis at 1000 evaluation points.")


All core functions defined and JIT-compiled.
Ready to proceed with analysis at 1000 evaluation points.


In [4]:

# There's a bug - the find_peaks returned fewer than 200 peaks
# I need to handle the case where we don't find enough peaks
# Let me fix the function

def analyze_N_optimized(N, func_name='zeta', n_t_points=1000, n_peaks=200, verbose=True):
 """Optimized analysis pipeline for a given truncation N"""
 if verbose:
 print(f"\n{'='*70}")
 print(f"Analyzing {func_name} at N={N:,}")
 print(f"{'='*70}")
 
 start_time = time.time()
 
 # Pre-compute omega and lambda
 if verbose:
 print(f"Computing ω(n) and Ω(n) for n ≤ {N:,}...")
 t0 = time.time()
 omega_vals, bigomega_vals = compute_omega_omega_bigomega(N)
 lambda_vals = ((-1)**bigomega_vals).astype(np.float64)
 k_max = int(np.max(omega_vals))
 if verbose:
 print(f" Done in {time.time()-t0:.1f}s. k_max = {k_max}")
 
 # Evaluate Dirichlet polynomial
 t_min = float(N)
 t_max = 2.0 * float(N)
 t_values = np.linspace(t_min, t_max, n_t_points)
 
 if verbose:
 print(f"Evaluating D_F(t; N) at {n_t_points:,} points...")
 t0 = time.time()
 
 if func_name == 'zeta':
 D_F = compute_dirichlet_zeta_jit(N, n_t_points)
 else: # liouville
 D_F = compute_dirichlet_liouville_jit(N, t_values, lambda_vals)
 
 modulus = np.abs(D_F)
 if verbose:
 print(f" Done in {time.time()-t0:.1f}s. |D_F| ∈ [{np.min(modulus):.2f}, {np.max(modulus):.2f}]")
 
 # Find peaks
 if verbose:
 print(f"Finding peaks...")
 t0 = time.time()
 peaks_idx, _ = find_peaks(modulus, prominence=1.0, distance=5)
 
 # Take top n_peaks or all peaks if fewer available
 peak_heights = modulus[peaks_idx]
 n_peaks_found = len(peaks_idx)
 n_peaks_to_use = min(n_peaks, n_peaks_found)
 
 top_idx = np.argsort(peak_heights)[-n_peaks_to_use:][::-1]
 top_peaks_idx = peaks_idx[top_idx]
 top_peaks_t = t_values[top_peaks_idx]
 top_peaks_height = modulus[top_peaks_idx]
 
 if verbose:
 print(f" Done in {time.time()-t0:.1f}s. Found {n_peaks_found} peaks, using top {n_peaks_to_use}")
 print(f" Peak heights ∈ [{np.min(top_peaks_height):.2f}, {np.max(top_peaks_height):.2f}]")
 
 # Compute r at each peak
 if verbose:
 print(f"Computing r values at {n_peaks_to_use} peaks...")
 t0 = time.time()
 
 r_values = np.zeros(n_peaks_to_use)
 
 for i, (t_peak, D_F_peak_idx) in enumerate(zip(top_peaks_t, top_peaks_idx)):
 D_F_peak = D_F[D_F_peak_idx]
 
 if func_name == 'zeta':
 S_k_real, S_k_imag = compute_omega_class_sums_zeta(N, t_peak, omega_vals, k_max)
 else:
 S_k_real, S_k_imag = compute_omega_class_sums_liouville(N, t_peak, omega_vals, lambda_vals, k_max)
 
 r_values[i] = compute_r_value(D_F_peak, S_k_real, S_k_imag)
 
 if verbose:
 print(f" Done in {time.time()-t0:.1f}s. Mean r = {np.mean(r_values):.4f}")
 
 # Quartile analysis
 if verbose:
 print(f"Computing quartile statistics...")
 
 q25, q50, q75 = np.percentile(top_peaks_height, [25, 50, 75])
 
 quartile_labels = np.ones(n_peaks_to_use, dtype=int)
 quartile_labels[top_peaks_height >= q25] = 2
 quartile_labels[top_peaks_height >= q50] = 3
 quartile_labels[top_peaks_height >= q75] = 4
 
 quartile_stats = {}
 for q in [1, 2, 3, 4]:
 mask = quartile_labels == q
 r_in_quartile = r_values[mask]
 if len(r_in_quartile) > 0:
 quartile_stats[f'Q{q}'] = {
 'mean_r': np.mean(r_in_quartile),
 'std_r': np.std(r_in_quartile),
 'n_peaks': np.sum(mask)
 }
 else:
 quartile_stats[f'Q{q}'] = {
 'mean_r': np.nan,
 'std_r': np.nan,
 'n_peaks': 0
 }
 
 if verbose:
 for q in [1, 2, 3, 4]:
 stats = quartile_stats[f'Q{q}']
 if stats['n_peaks'] > 0:
 print(f" Q{q}: mean_r = {stats['mean_r']:7.4f} ± {stats['std_r']:.4f} (n={stats['n_peaks']})")
 
 total_time = time.time() - start_time
 if verbose:
 print(f"Total time: {total_time:.1f}s ({total_time/60:.1f} min)")
 
 return {
 'N': N,
 'func_name': func_name,
 'r_values': r_values,
 'quartile_stats': quartile_stats,
 'peak_heights': top_peaks_height,
 'n_peaks_used': n_peaks_to_use,
 'total_time': total_time
 }

print("Fixed function - handling cases with fewer than requested peaks")


Fixed function - handling cases with fewer than requested peaks


In [5]:

# Retry N=10^4
all_results = {}

for func_name in ['zeta', 'liouville']:
 key = f'{func_name}_N10000'
 all_results[key] = analyze_N_optimized(10**4, func_name=func_name, n_t_points=1000, verbose=True)

print("\n" + "="*70)
print("N=10^4 completed for both functions")
print("="*70)



Analyzing zeta at N=10,000
Computing ω(n) and Ω(n) for n ≤ 10,000...
 Done in 0.1s. k_max = 5
Evaluating D_F(t; N) at 1,000 points...


 Done in 0.4s. |D_F| ∈ [0.00, 17.27]
Finding peaks...
 Done in 0.0s. Found 119 peaks, using top 119
 Peak heights ∈ [1.36, 17.27]
Computing r values at 119 peaks...
 Done in 0.1s. Mean r = 0.5444
Computing quartile statistics...
 Q1: mean_r = 0.2744 ± 0.4634 (n=30)
 Q2: mean_r = 0.4142 ± 0.4869 (n=29)
 Q3: mean_r = 0.5796 ± 0.5535 (n=30)
 Q4: mean_r = 0.9050 ± 0.4473 (n=30)
Total time: 0.5s (0.0 min)

Analyzing liouville at N=10,000
Computing ω(n) and Ω(n) for n ≤ 10,000...
 Done in 0.1s. k_max = 5
Evaluating D_F(t; N) at 1,000 points...


 Done in 0.5s. |D_F| ∈ [0.02, 21.83]
Finding peaks...
 Done in 0.0s. Found 109 peaks, using top 109
 Peak heights ∈ [1.77, 21.83]
Computing r values at 109 peaks...
 Done in 0.1s. Mean r = 0.6109
Computing quartile statistics...
 Q1: mean_r = 0.2259 ± 0.3508 (n=27)
 Q2: mean_r = 0.5864 ± 0.5351 (n=27)
 Q3: mean_r = 0.8421 ± 0.3858 (n=27)
 Q4: mean_r = 0.7828 ± 0.3080 (n=28)
Total time: 0.7s (0.0 min)

N=10^4 completed for both functions


In [6]:

# Good! Now continue with N=10^5
for func_name in ['zeta', 'liouville']:
 key = f'{func_name}_N100000'
 all_results[key] = analyze_N_optimized(10**5, func_name=func_name, n_t_points=1000, verbose=True)

print("\n" + "="*70)
print("N=10^5 completed for both functions")
print("="*70)



Analyzing zeta at N=100,000
Computing ω(n) and Ω(n) for n ≤ 100,000...


 Done in 1.1s. k_max = 6
Evaluating D_F(t; N) at 1,000 points...
 Done in 3.5s. |D_F| ∈ [0.00, 24.74]
Finding peaks...
 Done in 0.0s. Found 126 peaks, using top 126
 Peak heights ∈ [1.09, 24.74]
Computing r values at 126 peaks...


 Done in 0.4s. Mean r = 0.4563
Computing quartile statistics...
 Q1: mean_r = 0.0024 ± 0.2824 (n=32)
 Q2: mean_r = 0.2939 ± 0.4004 (n=31)
 Q3: mean_r = 0.6730 ± 0.5417 (n=31)
 Q4: mean_r = 0.8577 ± 0.5399 (n=32)
Total time: 5.0s (0.1 min)

Analyzing liouville at N=100,000
Computing ω(n) and Ω(n) for n ≤ 100,000...


 Done in 1.1s. k_max = 6
Evaluating D_F(t; N) at 1,000 points...
 Done in 3.5s. |D_F| ∈ [0.04, 33.92]
Finding peaks...
 Done in 0.0s. Found 123 peaks, using top 123
 Peak heights ∈ [1.41, 33.92]
Computing r values at 123 peaks...


 Done in 0.4s. Mean r = 0.4170
Computing quartile statistics...
 Q1: mean_r = 0.0463 ± 0.4645 (n=31)
 Q2: mean_r = 0.2972 ± 0.3506 (n=30)
 Q3: mean_r = 0.5215 ± 0.4019 (n=31)
 Q4: mean_r = 0.7991 ± 0.5193 (n=31)
Total time: 4.9s (0.1 min)

N=10^5 completed for both functions


In [7]:

# Continue with N=10^6
for func_name in ['zeta', 'liouville']:
 key = f'{func_name}_N1000000'
 all_results[key] = analyze_N_optimized(10**6, func_name=func_name, n_t_points=1000, verbose=True)

print("\n" + "="*70)
print("N=10^6 completed for both functions")
print("="*70)



Analyzing zeta at N=1,000,000
Computing ω(n) and Ω(n) for n ≤ 1,000,000...


 Done in 10.9s. k_max = 7
Evaluating D_F(t; N) at 1,000 points...
 Done in 34.1s. |D_F| ∈ [0.00, 24.61]
Finding peaks...
 Done in 0.0s. Found 139 peaks, using top 139
 Peak heights ∈ [1.43, 24.61]
Computing r values at 139 peaks...


 Done in 4.4s. Mean r = 0.3408
Computing quartile statistics...
 Q1: mean_r = -0.0071 ± 0.2883 (n=35)
 Q2: mean_r = 0.1650 ± 0.4108 (n=34)
 Q3: mean_r = 0.3346 ± 0.4952 (n=35)
 Q4: mean_r = 0.8656 ± 0.5618 (n=35)
Total time: 49.3s (0.8 min)

Analyzing liouville at N=1,000,000
Computing ω(n) and Ω(n) for n ≤ 1,000,000...


 Done in 10.9s. k_max = 7
Evaluating D_F(t; N) at 1,000 points...
 Done in 33.9s. |D_F| ∈ [0.03, 27.30]
Finding peaks...
 Done in 0.0s. Found 132 peaks, using top 132
 Peak heights ∈ [1.71, 27.30]
Computing r values at 132 peaks...


 Done in 4.2s. Mean r = 0.3721
Computing quartile statistics...
 Q1: mean_r = -0.0175 ± 0.2863 (n=33)
 Q2: mean_r = 0.2697 ± 0.4521 (n=33)
 Q3: mean_r = 0.5022 ± 0.4525 (n=33)
 Q4: mean_r = 0.7342 ± 0.5461 (n=33)
Total time: 49.0s (0.8 min)

N=10^6 completed for both functions


In [8]:

# The N=10^7 computation timed out. Let me work with what I have (N=10^4, 10^5, 10^6)
# and document this limitation clearly

print("ANALYSIS STATUS:")
print("="*70)
print("✓ N=10^4: COMPLETED (zeta, liouville)")
print("✓ N=10^5: COMPLETED (zeta, liouville)")
print("✓ N=10^6: COMPLETED (zeta, liouville)")
print("✗ N=10^7: TIMED OUT (computational limit exceeded)")
print("="*70)

# Extract and tabulate the results we have
import pandas as pd

# Create summary table
summary_data = []

for key, result in all_results.items():
 func_name = result['func_name']
 N = result['N']
 
 for q in ['Q1', 'Q2', 'Q3', 'Q4']:
 stats = result['quartile_stats'][q]
 summary_data.append({
 'Function': func_name,
 'N': N,
 'Quartile': q,
 'mean_r': stats['mean_r'],
 'std_r': stats['std_r'],
 'n_peaks': stats['n_peaks']
 })

df = pd.DataFrame(summary_data)

# Pivot for easier viewing
print("\nMean r values by N and Quartile:")
print("="*70)
for func in ['zeta', 'liouville']:
 print(f"\n{func.upper()}:")
 func_data = df[df['Function'] == func]
 pivot = func_data.pivot(index='Quartile', columns='N', values='mean_r')
 print(pivot.to_string())
 
print("\n" + "="*70)


TimeoutError: Code execution timed out after 568 seconds